In [ ]:
import pandas as pd
import numpy as np

# 1) โหลดข้อมูล
weather_path = "songkhla_weather_2020-merged.csv"
car_count_path = "car-count63.csv"

df = pd.read_csv(weather_path)
car = pd.read_csv(car_count_path)

# ตรวจว่ามีคอลัมน์ที่ต้องใช้
required_cols = {"ทางหลวงสาย","vehicles_lt_4_wheels","vehicles_4_wheels","vehicles_gt_4_wheels"}
missing = required_cols - set(car.columns)
if missing:
    raise ValueError(f"Columns missing in {car_count_path}: {missing}")

# หาค่ารวมหลักจาก car-count67: ถ้ามีแถว 'รวม' ใช้แถวนั้น ไม่งั้นรวมเฉพาะสาย 4
def coerce_numeric(x):
    try:
        return float(x)
    except Exception:
        return np.nan

car["_สาย_numeric_"] = car["ทางหลวงสาย"].apply(coerce_numeric)
has_total_row = car["_สาย_numeric_"].isna().any()

if has_total_row:
    total_row = car.loc[car["_สาย_numeric_"].isna()].iloc[-1]
    GRAND_SMALL = int(total_row["vehicles_lt_4_wheels"])
    GRAND_FOUR  = int(total_row["vehicles_4_wheels"])
    GRAND_HEAVY = int(total_row["vehicles_gt_4_wheels"])
else:
    subset = car[car["_สาย_numeric_"] == 4]
    GRAND_SMALL = int(subset["vehicles_lt_4_wheels"].sum())
    GRAND_FOUR  = int(subset["vehicles_4_wheels"].sum())
    GRAND_HEAVY = int(subset["vehicles_gt_4_wheels"].sum())

print("Grand totals (จะกระจายทั้งชุดให้เท่านี้เป๊ะ):")
print({"small": GRAND_SMALL, "four": GRAND_FOUR, "heavy": GRAND_HEAVY})

# 2) แปลงเวลา (AM/PM) + เตรียม condition
# รวม date + time เป็น datetime ชัดเจน (รองรับ AM/PM)
dt_str = (df["date"].astype(str).str.strip() + " " +
          df["time"].astype(str).str.strip())
try:
    df["datetime"] = pd.to_datetime(dt_str, format="%m/%d/%Y %I:%M %p", errors="raise")
except Exception:
    # fallback ถ้าบางแถว format แปลก
    df["datetime"] = pd.to_datetime(dt_str, errors="coerce")

bad = df["datetime"].isna().sum()
if bad:
    raise ValueError(f"พบแถวเวลาแปลงไม่ได้ {bad} แถว — โปรดตรวจไฟล์ต้นทาง")

df["hour"] = df["datetime"].dt.hour

def normalize_condition(c):
    c = str(c).strip().lower()
    if "rain" in c or "shower" in c or "storm" in c:
        return "Rainy"
    if "mist" in c or "fog" in c:
        return "Mist"
    return "Clear"

df["cond_norm"] = df["condition"].apply(normalize_condition)

# 3) นิยามสัดส่วนตามช่วงเวลา (ทั้งวันรวม=1) + ตัวคูณตามอากาศ
def time_block(h):
    if 0 <= h <= 4:
        return "late_night"
    if 5 <= h <= 8:
        return "morning"
    if 9 <= h <= 15:
        return "mid_day"
    if 16 <= h <= 19:
        return "evening"
    return "night"  # 20-23

BLOCK_FACTORS_SMALL = {
    "late_night": 0.05,
    "morning":    0.25,
    "mid_day":    0.30,
    "evening":    0.25,
    "night":      0.15,
}
BLOCK_FACTORS_FOUR = {
    "late_night": 0.10,
    "morning":    0.35,
    "mid_day":    0.30,
    "evening":    0.35,
    "night":      0.20,
}
BLOCK_FACTORS_HEAVY = {
    "late_night": 0.25,
    "morning":    0.15,
    "mid_day":    0.25,
    "evening":    0.20,
    "night":      0.15,
}

WEATHER_MULT = {
    "Clear":  {"small": 1.00, "four": 1.00, "heavy": 1.00},
    "Mist": {"small": 0.90, "four": 1.00, "heavy": 1.00},
    "Rainy":  {"small": 0.70, "four": 1.10, "heavy": 1.00},
}

# 4) คำนวณ weight รายแถว "ทั้งชุด" (ไม่แยกรายวัน)
hours = df["hour"].to_numpy()
conds = df["cond_norm"].to_numpy()

# สร้าง weight base ต่อแถวจาก block ของชั่วโมง
def block_weight_for_series(hours, block_table):
    # คำนวณ weight ของ block ให้ “เฉลี่ยต่อชั่วโมงในบล็อค” แล้ว normalize ทั้งชุด
    blocks = np.array([time_block(h) for h in hours], dtype=object)
    unique, counts = np.unique(blocks, return_counts=True)
    per_block_len = {b: c for b, c in zip(unique, counts)}
    w = np.array([block_table[b] / per_block_len[b] for b in blocks], dtype=float)
    # normalize ทั้งชุด
    w /= w.sum()
    return w

w_small = block_weight_for_series(hours, BLOCK_FACTORS_SMALL)
w_four  = block_weight_for_series(hours, BLOCK_FACTORS_FOUR)
w_heavy = block_weight_for_series(hours, BLOCK_FACTORS_HEAVY)

# ปรับด้วยสภาพอากาศต่อแถว
mult_small = np.array([WEATHER_MULT.get(c, WEATHER_MULT["Clear"])["small"] for c in conds], dtype=float)
mult_four  = np.array([WEATHER_MULT.get(c, WEATHER_MULT["Clear"])["four"]  for c in conds], dtype=float)
mult_heavy = np.array([WEATHER_MULT.get(c, WEATHER_MULT["Clear"])["heavy"] for c in conds], dtype=float)

w_small *= mult_small
w_four  *= mult_four
w_heavy *= mult_heavy

# ใส่ noise เล็กน้อยเพื่อความสมจริง (เฉพาะชุด APPROX)
rng = np.random.default_rng(12345)
def with_noise(w, jitter=0.10):
    n = rng.normal(0, jitter, size=w.shape)
    w2 = np.clip(w * (1 + n), 1e-12, None)
    return w2

w_small_approx = with_noise(w_small, 0.10)
w_four_approx  = with_noise(w_four,  0.10)
w_heavy_approx = with_noise(w_heavy, 0.10)

# normalize weights อีกครั้งให้รวม=1 (กันไม่ให้เกิน)
def normalize(w):
    s = w.sum()
    if s <= 0:
        return np.ones_like(w) / len(w)
    return w / s

w_small      = normalize(w_small)
w_four       = normalize(w_four)
w_heavy      = normalize(w_heavy)
w_small_apx  = normalize(w_small_approx)
w_four_apx   = normalize(w_four_approx)
w_heavy_apx  = normalize(w_heavy_approx)

# 5) ตัวช่วยจัดสรรจำนวนเต็ม “รวมเท่ากับ GRAND” (ไม่เกินแน่)
def allocate_integers_from_weights(weights, total):
    """Largest remainder allocation — รวมเท่ากับ total, ไม่เกิน"""
    weights = np.array(weights, dtype=float)
    weights = normalize(weights)
    raw = weights * total
    base = np.floor(raw).astype(int)
    remainder = int(total - base.sum())
    if remainder > 0:
        frac = raw - base
        idx = np.argsort(-frac)[:remainder]
        base[idx] += 1
    return base

def multinomial_exact(weights, total, seed=2024):
    rng2 = np.random.default_rng(seed)
    weights = normalize(weights)
    return rng2.multinomial(int(total), weights)

# 6) แจกแจง “ทั้งชุด” → ได้จำนวนเต็มรายแถว
# APPROX (มี noise)
small_apx = allocate_integers_from_weights(w_small_apx, GRAND_SMALL)
four_apx  = allocate_integers_from_weights(w_four_apx,  GRAND_FOUR)
heavy_apx = allocate_integers_from_weights(w_heavy_apx, GRAND_HEAVY)

# EXACT (multinomial)
small_ex  = multinomial_exact(w_small, GRAND_SMALL, seed=999)
four_ex   = multinomial_exact(w_four,  GRAND_FOUR,  seed=999)
heavy_ex  = multinomial_exact(w_heavy, GRAND_HEAVY, seed=999)

# ตรวจรวม (ต้องเท่าค่ารวมหลักเป๊ะ)
assert small_apx.sum() == GRAND_SMALL and four_apx.sum() == GRAND_FOUR and heavy_apx.sum() == GRAND_HEAVY
assert small_ex.sum()  == GRAND_SMALL and four_ex.sum()  == GRAND_FOUR  and heavy_ex.sum()  == GRAND_HEAVY

# 7) เขียนผลลัพธ์
df_approx = df.copy()
df_exact  = df.copy()

df_approx["vehicles_lt_4_wheels"] = small_apx
df_approx["vehicles_4_wheels"]        = four_apx
df_approx["vehicles_gt_4_wheels"]  = heavy_apx

df_exact["vehicles_lt_4_wheels"] = small_ex
df_exact["vehicles_4_wheels"]        = four_ex
df_exact["vehicles_gt_4_wheels"]  = heavy_ex

# เผื่อบางคอลัมน์อุบัติเหตุไม่มีในไฟล์ต้นทาง ให้เติมศูนย์
for c in ["เกิดเหตุ","รถน้อยกว่า4ล้อacc","รถ4ล้อacc","รถมากกว่า4ล้อacc"]:
    if c not in df_approx.columns:
        df_approx[c] = 0
        df_exact[c]  = 0

cols_out = [
    "date","time","temperature_F","humidity_%","pressure_in","condition",
    "เกิดเหตุ","รถน้อยกว่า4ล้อacc","รถ4ล้อacc","รถมากกว่า4ล้อacc",
    "vehicles_lt_4_wheels","vehicles_4_wheels","vehicles_gt_4_wheels"
]

df_approx[cols_out].to_csv("datafinal-2020.csv", index=False, encoding="utf-8-sig")

print("Saved: acc_weather_with_traffic_APPROX_GRAND.csv")
print("Saved: acc_weather_with_traffic_EXACT_GRAND.csv")
print("   - ทั้งสองไฟล์รวมทั้งชุด = ค่ารวมหลักจาก car-count67.csv")


Grand totals (จะกระจายทั้งชุดให้เท่านี้เป๊ะ):
{'small': 39344, 'four': 155491, 'heavy': 30994}


C:\Users\sakka\AppData\Local\Temp\ipykernel_12296\3121382569.py:49: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["datetime"] = pd.to_datetime(dt_str, errors="coerce")


Saved: acc_weather_with_traffic_APPROX_GRAND.csv
Saved: acc_weather_with_traffic_EXACT_GRAND.csv
   - ทั้งสองไฟล์รวมทั้งชุด = ค่ารวมหลักจาก car-count67.csv
